In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

matches = pd.read_csv('clean_matches.csv')
deliveries = pd.read_csv('clean_deliveries.csv')

matches.columns = matches.columns.str.lower().str.strip()
deliveries.columns = deliveries.columns.str.lower().str.strip()

matches['matchid'] = matches['matchid'].astype(str).str.strip()
deliveries['matchid'] = deliveries['matchid'].astype(str).str.strip()

if 'date' in matches.columns:
    matches['date'] = pd.to_datetime(matches['date'], errors='coerce')

merge_cols = [c for c in ['matchid','venue','season','date','team1','team2','winner'] if c in matches.columns]
deliveries = deliveries.merge(matches[merge_cols], on='matchid', how='left')

for col, default in [
    ('venue', 'Unknown'),
    ('season', 'Unknown'),
    ('team1', 'Unknown'),
    ('team2', 'Unknown'),
    ('winner', 'No Result')
]:
    if col in deliveries.columns:
        deliveries[col] = deliveries[col].fillna(default)
    else:
        deliveries[col] = default

if 'date' in deliveries.columns:
    deliveries['date'] = pd.to_datetime(deliveries['date'], errors='coerce')
else:
    deliveries['date'] = pd.NaT

In [3]:
valid = deliveries[deliveries.get('is_valid_ball', 0) == 1]

innings = valid.groupby(['matchid', 'inning', 'batsman']).agg(
    runs_scored=('batsman_runs', 'sum'),
    balls_faced=('ball', 'count')
).reset_index()

info_cols = ['matchid', 'inning', 'batsman']
for c in ['batting_team', 'bowling_team', 'venue', 'season', 'date', 'team1', 'team2']:
    if c in deliveries.columns:
        info_cols.append(c)

info = deliveries[info_cols].drop_duplicates(subset=['matchid', 'inning', 'batsman'])

innings = innings.merge(info, on=['matchid', 'inning', 'batsman'], how='left')

innings['strike_rate'] = np.where(
    innings['balls_faced'] > 0,
    innings['runs_scored'] / innings['balls_faced'] * 100,
    0
)

if 'date' in innings.columns:
    innings['date'] = pd.to_datetime(innings['date'], errors='coerce')
    innings = innings.sort_values(['batsman', 'date']).reset_index(drop=True)
else:
    innings = innings.sort_values(['batsman', 'matchid']).reset_index(drop=True)

print("Batsman-innings rows:", len(innings))
print("Columns:", innings.columns.tolist())

Batsman-innings rows: 4196
Columns: ['matchid', 'inning', 'batsman', 'runs_scored', 'balls_faced', 'batting_team', 'bowling_team', 'venue', 'season', 'date', 'team1', 'team2', 'strike_rate']


In [4]:
innings['innings_played'] = innings.groupby('batsman').cumcount() + 1

innings['batsman_avg'] = innings.groupby('batsman')['runs_scored'].transform(
    lambda x: x.shift(1).expanding().mean().fillna(0)
)

innings['balls_faced_avg'] = innings.groupby('batsman')['balls_faced'].transform(
    lambda x: x.shift(1).expanding().mean().fillna(0)
)

cum_r = innings.groupby('batsman')['runs_scored'].cumsum().shift(1).fillna(0)
cum_b = innings.groupby('batsman')['balls_faced'].cumsum().shift(1).fillna(0)
innings['historical_sr'] = np.where(cum_b > 0, cum_r / cum_b * 100, 0)

temp = innings.sort_values(['batsman', 'bowling_team', 'matchid'])
temp['avg_vs_opp'] = temp.groupby(['batsman', 'bowling_team'])['runs_scored'].transform(
    lambda x: x.shift(1).expanding().mean().fillna(0)
)
innings = temp.sort_values(['batsman', 'matchid']).reset_index(drop=True)

In [5]:
innings_tot = deliveries.groupby(['matchid', 'inning'])['total_runs'].sum().reset_index(name='innings_total')

innings_tot = innings_tot.merge(
    deliveries[['matchid', 'inning', 'bowling_team']].drop_duplicates(),
    on=['matchid', 'inning']
)

innings_tot['opp_strength'] = innings_tot.groupby('bowling_team')['innings_total'].transform(
    lambda x: x.shift(1).expanding().mean().fillna(innings_tot['innings_total'].mean())
)

innings = innings.merge(
    innings_tot[['matchid', 'inning', 'opp_strength']],
    on=['matchid', 'inning'],
    how='left'
)

innings['opp_strength'] = innings['opp_strength'].fillna(innings['opp_strength'].mean())

if 'team1' in innings.columns:
    innings['home_advantage'] = (innings['batting_team'] == innings['team1']).astype(int)
else:
    innings['home_advantage'] = 0

if 'season' in innings.columns:
    innings['season_year'] = innings['season'].astype(str).str.extract(r'(\d{4})').astype(float)
    innings['season_year'] = innings['season_year'].fillna(innings['date'].dt.year if 'date' in innings.columns else 2020)
else:
    innings['season_year'] = 2020

le = LabelEncoder()
innings['venue_encoded'] = le.fit_transform(innings['venue'])

final = innings[innings['innings_played'] > 5].copy()

final.to_csv('processed_player_data.csv', index=False)
import joblib
joblib.dump(le, 'venue_encoder.pkl')

print("\nSaved processed_player_data.csv and venue_encoder.pkl")
print("Final rows:", len(final))
print("Final columns:", final.columns.tolist())
print("\nSample:\n", final[['batsman','runs_scored','batsman_avg','historical_sr','avg_vs_opp','venue_encoded','opp_strength','home_advantage','season_year']].head(3))


Saved processed_player_data.csv and venue_encoder.pkl
Final rows: 2732
Final columns: ['matchid', 'inning', 'batsman', 'runs_scored', 'balls_faced', 'batting_team', 'bowling_team', 'venue', 'season', 'date', 'team1', 'team2', 'strike_rate', 'innings_played', 'batsman_avg', 'balls_faced_avg', 'historical_sr', 'avg_vs_opp', 'opp_strength', 'home_advantage', 'season_year', 'venue_encoded']

Sample:
      batsman  runs_scored  batsman_avg  historical_sr  avg_vs_opp  \
5   A Chopra         11.0          8.4      80.769231        24.0   
14  A Kumble          1.0          2.8      63.636364         0.0   
15  A Kumble          5.0          2.5      65.217391         1.5   

    venue_encoded  opp_strength  home_advantage  season_year  
5               0    164.714286               0          NaN  
14              0    160.466667               0          NaN  
15              0    148.454545               0          NaN  
